# Online Retail: Data Cleaning, EDA & Business Insights
### End-to-End Data Analysis Project | Data Analysis & Data Handling

**Objective:** Clean and validate a real retail transactions dataset, explore sales patterns, create visualizations, and produce evidence-based business insights.

**Dataset:** UCI Online Retail  
**Source:** https://archive.ics.uci.edu/dataset/352/online+retail  
**Original coverage:** 1 December 2010 – 9 December 2011  
**License:** CC BY 4.0 — attribute the dataset source when sharing results.



## 1. Setup and Dataset Collection

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
sns.set_theme(style="whitegrid")

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
DATA_FILE = Path("Online_Retail.xlsx")

if not DATA_FILE.exists():
    urlretrieve(DATA_URL, DATA_FILE)

raw_df = pd.read_excel(DATA_FILE, engine="openpyxl")
raw_df.head()

### Inspect the dataset dimensions and column names.

In [ ]:
raw_df.shape, raw_df.columns.tolist()

### Inspect data types, missing values, and sample records.

In [ ]:
raw_df.info()

### View a random sample of transaction rows.

In [ ]:
raw_df.sample(min(8, len(raw_df)), random_state=42)

## 2. Understand the Data and Define Quality Checks

The dataset contains transaction line items. An invoice may contain multiple products, so a row is not necessarily a complete order. The `InvoiceNo` values beginning with `C` indicate cancellations. Negative quantities can represent returns; they should not be silently treated as ordinary sales.

### Create a data-quality summary before cleaning.

In [ ]:
quality_before = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing_count": raw_df.isna().sum(),
    "missing_percent": (raw_df.isna().mean() * 100).round(2),
    "unique_values": raw_df.nunique(dropna=True)
}).sort_values("missing_percent", ascending=False)
quality_before

### Count duplicate rows, cancellation invoices, non-positive quantities, and non-positive prices.

In [ ]:
quality_flags = pd.Series({
    "raw_rows": len(raw_df),
    "duplicate_rows": int(raw_df.duplicated().sum()),
    "missing_customer_id_rows": int(raw_df["CustomerID"].isna().sum()),
    "cancellation_line_rows": int(raw_df["InvoiceNo"].astype(str).str.startswith("C", na=False).sum()),
    "quantity_zero_or_negative_rows": int((raw_df["Quantity"] <= 0).sum()),
    "unit_price_zero_or_negative_rows": int((raw_df["UnitPrice"] <= 0).sum()),
    "missing_description_rows": int(raw_df["Description"].isna().sum()),
    "missing_invoice_date_rows": int(raw_df["InvoiceDate"].isna().sum())
})
quality_flags

## 3. Clean and Validate the Data

### Cleaning rules used for the main sales analysis

1. Keep a separate raw DataFrame; do not overwrite the source.
2. Remove exact duplicate line records.
3. Convert invoice date to datetime and key numeric fields to numeric types.
4. Exclude cancellation invoices and non-positive quantity or unit-price rows from the **positive-sales analysis**.
5. Exclude rows without `CustomerID` only from customer-level analyses; retain a sales-cleaned dataset for transaction and country analysis.
6. Recalculate `Revenue = Quantity × UnitPrice`.
7. Treat these filters as an analytical definition, not a claim that all excluded rows are erroneous. Returns/cancellations should be analyzed separately when the business question requires it.

In [ ]:
clean_df = raw_df.copy()

clean_df = clean_df.drop_duplicates().copy()
clean_df["InvoiceDate"] = pd.to_datetime(clean_df["InvoiceDate"], errors="coerce")
clean_df["Quantity"] = pd.to_numeric(clean_df["Quantity"], errors="coerce")
clean_df["UnitPrice"] = pd.to_numeric(clean_df["UnitPrice"], errors="coerce")
clean_df["CustomerID"] = pd.to_numeric(clean_df["CustomerID"], errors="coerce")

clean_df = clean_df.dropna(subset=["InvoiceNo", "StockCode", "InvoiceDate", "Quantity", "UnitPrice", "Country"])
clean_df["InvoiceNo"] = clean_df["InvoiceNo"].astype(str)
clean_df["StockCode"] = clean_df["StockCode"].astype(str)
clean_df["Country"] = clean_df["Country"].astype(str).str.strip()
clean_df["Description"] = clean_df["Description"].astype("string").str.strip()

cancellation_mask = clean_df["InvoiceNo"].str.upper().str.startswith("C")
positive_sales_mask = (~cancellation_mask) & (clean_df["Quantity"] > 0) & (clean_df["UnitPrice"] > 0)

sales_df = clean_df.loc[positive_sales_mask].copy()
sales_df["Revenue"] = sales_df["Quantity"] * sales_df["UnitPrice"]
sales_df["InvoiceDate"] = pd.to_datetime(sales_df["InvoiceDate"])

sales_df.head()

### Validate the cleaned sales data.

In [ ]:
validation_summary = pd.Series({
    "rows_before_cleaning": len(raw_df),
    "rows_after_exact_deduplication_and_type_checks": len(clean_df),
    "positive_sales_rows": len(sales_df),
    "remaining_duplicate_rows": int(sales_df.duplicated().sum()),
    "missing_invoice_dates": int(sales_df["InvoiceDate"].isna().sum()),
    "non_positive_quantities": int((sales_df["Quantity"] <= 0).sum()),
    "non_positive_unit_prices": int((sales_df["UnitPrice"] <= 0).sum()),
    "non_positive_revenue_rows": int((sales_df["Revenue"] <= 0).sum())
})
validation_summary

### Compare rows excluded by each major cleaning rule.

In [ ]:
exclusion_summary = pd.Series({
    "exact_duplicates_in_raw": int(raw_df.duplicated().sum()),
    "cancellation_rows_after_deduplication": int(clean_df["InvoiceNo"].str.upper().str.startswith("C").sum()),
    "non_positive_quantity_rows_after_deduplication": int((clean_df["Quantity"] <= 0).sum()),
    "non_positive_unit_price_rows_after_deduplication": int((clean_df["UnitPrice"] <= 0).sum()),
    "missing_customer_ids_in_sales_data": int(sales_df["CustomerID"].isna().sum())
})
exclusion_summary

## 4. Exploratory Data Analysis

### Summarize the analysis-ready sales dataset.

In [ ]:
sales_df[['Quantity', 'UnitPrice', 'Revenue']].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])

### Count distinct invoices, products, countries, and identified customers.

In [ ]:
pd.Series({
    "transaction_lines": len(sales_df),
    "unique_invoices": sales_df["InvoiceNo"].nunique(),
    "unique_products": sales_df["StockCode"].nunique(),
    "countries": sales_df["Country"].nunique(),
    "identified_customers": sales_df["CustomerID"].nunique()
})

### Calculate total positive sales revenue in GBP.

In [ ]:
sales_df['Revenue'].sum()

### Identify the overall date coverage of the cleaned sales data.

In [ ]:
pd.Series({'first_transaction': sales_df['InvoiceDate'].min(), 'last_transaction': sales_df['InvoiceDate'].max()})

## 5. Sales Trends Over Time

### Calculate monthly revenue and invoice count.

In [ ]:
monthly_sales = (
    sales_df.assign(Month=sales_df["InvoiceDate"].dt.to_period("M").dt.to_timestamp())
    .groupby("Month")
    .agg(revenue=("Revenue", "sum"), invoices=("InvoiceNo", "nunique"), units=("Quantity", "sum"))
    .sort_index()
)
monthly_sales

### Visualize monthly revenue.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(monthly_sales.index, monthly_sales["revenue"], marker="o")
ax.set(title="Monthly Positive Sales Revenue", xlabel="Month", ylabel="Revenue (GBP)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### Compare monthly invoice counts.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(monthly_sales.index.strftime("%Y-%m"), monthly_sales["invoices"])
ax.set(title="Unique Invoices by Month", xlabel="Month", ylabel="Invoice Count")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### Find the month with the highest revenue and the month with the lowest revenue.

In [ ]:
pd.DataFrame({
    "highest_revenue_month": [monthly_sales["revenue"].idxmax()],
    "highest_revenue_gbp": [monthly_sales["revenue"].max()],
    "lowest_revenue_month": [monthly_sales["revenue"].idxmin()],
    "lowest_revenue_gbp": [monthly_sales["revenue"].min()]
})

**Interpretation caution:** The dataset ends on 9 December 2011, so the final month is incomplete. Do not compare its raw monthly total with complete months without noting this coverage limitation.

### Compare revenue by day of week.

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_sales = sales_df.assign(Weekday=sales_df["InvoiceDate"].dt.day_name()).groupby("Weekday")["Revenue"].sum().reindex(weekday_order)
weekday_sales

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
weekday_sales.plot(kind="bar", ax=ax)
ax.set(title="Positive Sales Revenue by Day of Week", xlabel="Day", ylabel="Revenue (GBP)")
plt.tight_layout()
plt.show()

## 6. Country and Product Analysis

### Find the top 10 countries by positive sales revenue.

In [ ]:
country_sales = sales_df.groupby("Country").agg(
    revenue=("Revenue", "sum"),
    invoices=("InvoiceNo", "nunique"),
    units=("Quantity", "sum")
).sort_values("revenue", ascending=False)
country_sales.head(10)

### Visualize the top 10 countries by revenue.

In [ ]:
top_countries = country_sales.head(10).sort_values("revenue")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_countries.index, top_countries["revenue"])
ax.set(title="Top 10 Countries by Positive Sales Revenue", xlabel="Revenue (GBP)", ylabel="Country")
plt.tight_layout()
plt.show()

### Find the top 10 products by revenue.

In [ ]:
product_sales = sales_df.groupby(["StockCode", "Description"], dropna=False).agg(
    revenue=("Revenue", "sum"),
    units=("Quantity", "sum"),
    invoice_count=("InvoiceNo", "nunique")
).sort_values("revenue", ascending=False)
product_sales.head(10)

### Visualize the top 10 products by revenue.

In [ ]:
top_products = product_sales.head(10).reset_index().sort_values("revenue")
top_products["product_label"] = top_products["StockCode"] + " — " + top_products["Description"].fillna("Unknown").str.slice(0, 35)
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top_products["product_label"], top_products["revenue"])
ax.set(title="Top 10 Products by Positive Sales Revenue", xlabel="Revenue (GBP)", ylabel="Product")
plt.tight_layout()
plt.show()

### Calculate the share of revenue contributed by the top country.

In [ ]:
top_country_share = country_sales["revenue"].max() / country_sales["revenue"].sum() * 100
pd.Series({"top_country": country_sales["revenue"].idxmax(), "top_country_revenue_share_percent": top_country_share})

## 7. Customer-Level Analysis

### Build a customer-level summary for rows with a known CustomerID.

In [ ]:
customer_sales = sales_df.dropna(subset=["CustomerID"]).copy()
customer_sales["CustomerID"] = customer_sales["CustomerID"].astype("Int64")
customer_summary = customer_sales.groupby("CustomerID").agg(
    revenue=("Revenue", "sum"),
    invoices=("InvoiceNo", "nunique"),
    units=("Quantity", "sum"),
    last_purchase=("InvoiceDate", "max")
).sort_values("revenue", ascending=False)
customer_summary.head(10)

### Visualize the distribution of customer revenue.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(customer_summary["revenue"], bins=50, edgecolor="white")
ax.set(title="Customer Revenue Distribution", xlabel="Customer Revenue (GBP)", ylabel="Customer Count")
plt.tight_layout()
plt.show()

### Calculate the share of revenue contributed by the top 10 customers.

In [ ]:
customer_revenue_share = customer_summary.head(10)["revenue"].sum() / customer_summary["revenue"].sum() * 100
pd.Series({"top_10_customer_revenue_share_percent": customer_revenue_share})

## 8. Distribution and Outlier Checks

### Visualize the distribution of transaction-line revenue.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sales_df["Revenue"].clip(upper=sales_df["Revenue"].quantile(0.99)), bins=50, edgecolor="white")
ax.set(title="Transaction-Line Revenue (Capped at 99th Percentile for Display)", xlabel="Revenue (GBP)", ylabel="Line Count")
plt.tight_layout()
plt.show()

### Detect high transaction-line revenue values using the IQR rule.

In [ ]:
q1 = sales_df["Revenue"].quantile(0.25)
q3 = sales_df["Revenue"].quantile(0.75)
iqr = q3 - q1
upper_iqr_bound = q3 + 1.5 * iqr
pd.Series({
    "q1_gbp": q1,
    "q3_gbp": q3,
    "iqr_gbp": iqr,
    "upper_iqr_bound_gbp": upper_iqr_bound,
    "rows_above_upper_bound": int((sales_df["Revenue"] > upper_iqr_bound).sum()),
    "percent_above_upper_bound": float((sales_df["Revenue"] > upper_iqr_bound).mean() * 100)
})

**Interpretation caution:** Large values may be legitimate wholesale purchases, not data errors. The chart caps values only for display; the underlying revenue values remain unchanged.

### Visualize the relationship between quantity and unit price on a sample.

In [ ]:
plot_sample = sales_df.sample(min(5000, len(sales_df)), random_state=42)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(plot_sample["Quantity"], plot_sample["UnitPrice"], alpha=0.25, s=12)
ax.set(title="Quantity vs Unit Price (Sample)", xlabel="Quantity", ylabel="Unit Price (GBP)")
plt.tight_layout()
plt.show()

## 9. Evidence-Based Insights (Generated from the Data)

The following 8 insights are computed from the loaded dataset and the cleaning rules above. Their numeric values should be read from the outputs after running the notebook; they are not hard-coded claims.

### Insight 1: Overall positive-sales revenue and transaction scale.

In [ ]:
pd.Series({
    "positive_sales_revenue_gbp": sales_df["Revenue"].sum(),
    "transaction_lines": len(sales_df),
    "unique_invoices": sales_df["InvoiceNo"].nunique(),
    "identified_customers": sales_df["CustomerID"].nunique()
})

### Insight 2: Highest-revenue month (with incomplete-period caution).

In [ ]:
monthly_sales.loc[monthly_sales['revenue'].idxmax()].to_frame('value')

### Insight 3: Country contributing the largest share of revenue.

In [ ]:
pd.Series({
    "top_country": country_sales["revenue"].idxmax(),
    "top_country_revenue_gbp": country_sales["revenue"].max(),
    "top_country_share_percent": country_sales["revenue"].max() / country_sales["revenue"].sum() * 100
})

### Insight 4: Products generating the most positive-sales revenue.

In [ ]:
product_sales.head(5)

### Insight 5: Revenue concentration among the top 10 products.

In [ ]:
top_10_product_share = product_sales.head(10)["revenue"].sum() / sales_df["Revenue"].sum() * 100
pd.Series({"top_10_product_revenue_share_percent": top_10_product_share})

### Insight 6: Revenue concentration among the top 10 identified customers.

In [ ]:
pd.Series({'top_10_customer_revenue_share_percent': customer_revenue_share})

### Insight 7: Data cleaning impact and missing customer identifiers.

In [ ]:
pd.Series({
    "raw_row_count": len(raw_df),
    "positive_sales_row_count": len(sales_df),
    "rows_not_in_positive_sales_analysis": len(raw_df) - len(sales_df),
    "missing_customer_ids_in_positive_sales": int(sales_df["CustomerID"].isna().sum()),
    "missing_customer_id_share_percent": float(sales_df["CustomerID"].isna().mean() * 100)
})

### Insight 8: Highest-revenue day of the week.

In [ ]:
pd.Series({
    "highest_revenue_weekday": weekday_sales.idxmax(),
    "weekday_revenue_gbp": weekday_sales.max(),
    "weekday_revenue_share_percent": weekday_sales.max() / weekday_sales.sum() * 100
})

### Insight 9: Outlier prevalence in transaction-line revenue.

In [ ]:
pd.Series({
    "upper_iqr_bound_gbp": upper_iqr_bound,
    "outlier_rows": int((sales_df["Revenue"] > upper_iqr_bound).sum()),
    "outlier_share_percent": float((sales_df["Revenue"] > upper_iqr_bound).mean() * 100)
})

### Insight 10: Highest-revenue country compared with the second-highest.

In [ ]:
country_top_two = country_sales.head(2)
pd.Series({
    "top_country": country_top_two.index[0],
    "second_country": country_top_two.index[1],
    "top_to_second_revenue_ratio": country_top_two["revenue"].iloc[0] / country_top_two["revenue"].iloc[1]
})

## 10. Final Data Quality Summary and Export

### Summarize the final analysis-ready dataset.

In [ ]:
final_summary = pd.Series({
    "analysis_rows": len(sales_df),
    "analysis_columns": sales_df.shape[1],
    "duplicate_rows": int(sales_df.duplicated().sum()),
    "missing_invoice_dates": int(sales_df["InvoiceDate"].isna().sum()),
    "non_positive_quantity_rows": int((sales_df["Quantity"] <= 0).sum()),
    "non_positive_unit_price_rows": int((sales_df["UnitPrice"] <= 0).sum()),
    "revenue_reconciliation_difference": float((sales_df["Revenue"] - sales_df["Quantity"] * sales_df["UnitPrice"]).abs().sum())
})
final_summary

### Export the cleaned transaction dataset.

In [ ]:
sales_df.to_csv("online_retail_cleaned_sales.csv", index=False)

### Export monthly sales summary.

In [ ]:
monthly_sales.to_csv("online_retail_monthly_summary.csv")

## Conclusion and Limitations

- This notebook uses a real, publicly documented transaction dataset and performs cleaning, validation, EDA, visualizations, and ten data-derived insight calculations.
- Results describe the selected positive-sales rows, not all cash movements: cancellations and returns are excluded from the main revenue analysis.
- The final month is incomplete, so month-to-month comparisons should account for observation coverage.
- High-value lines may reflect wholesale activity; outlier detection is a flag, not an automatic deletion rule.
- Customer-level summaries exclude rows with missing customer IDs; country and transaction-level summaries can still include those rows.
- These are descriptive associations. They do not establish causality.

## Dataset Citation

Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33

Dataset source and metadata: https://archive.ics.uci.edu/dataset/352/online+retail

**Next step:** Run all cells from top to bottom, review the computed insights, and then commit this notebook to GitHub. The dataset workbook is downloaded when the notebook runs and does not need to be included in the repository.